# Silver Layer: Passenger Table Cleaning & Validation

Cleanses `skyledger_airlines.bronze.passenger` into `skyledger_airlines.silver.passenger`.
Applies primary key validation (§3.1) and referential integrity checks (§3.4).
Invalid records are quarantined in `silver.passenger_quarantine` with comma-separated rejection reasons.
Includes Bronze-Silver-Quarantine reconciliation and a data-quality report.

**Relationships:** `passenger` → `booking` → `flight_booking` (core booking fact chain).
`nation` → `passenger` (passenger geography — home nation). FK `pa_nationkey` validated against `silver.nation`.

**No §3.2 numeric or §3.3 date checks apply** to the passenger table.

In [0]:
%sql
select * from skyledger_airlines.bronze.passenger

In [0]:
%sql
select * from skyledger_airlines.silver.passenger where pa_loyaltypoints < 0 

In [0]:
# ── Step 1: Read Bronze, Cast Types, Drop Metadata Columns ──
from pyspark.sql.functions import col, when, lit, count as _count, concat_ws
from pyspark.sql.window import Window

bronze_passenger = spark.table("skyledger_airlines.bronze.passenger")

# Drop metadata columns: _ingestion_timestamp, _source_file, _pipeline_run_id
# Cast key columns to INT, loyalty points to DOUBLE
passenger_typed = bronze_passenger.select(
    col("pa_passengerid").cast("int").alias("pa_passengerid"),
    col("pa_name").cast("string").alias("pa_name"),
    col("pa_address").cast("string").alias("pa_address"),
    col("pa_nationkey").cast("int").alias("pa_nationkey"),
    col("pa_phone").cast("string").alias("pa_phone"),
    col("pa_loyaltypoints").cast("double").alias("pa_loyaltypoints"),
    col("pa_travelpurpose").cast("string").alias("pa_travelpurpose"),
    col("pa_comment").cast("string").alias("pa_comment"),
)

bronze_count = passenger_typed.count()
print(f"Bronze passenger rows read: {bronze_count}")
display(passenger_typed.limit(10))

In [0]:
# ── Step 2: Primary Key Validation (§3.1) ──
# Rule: pa_passengerid must be NOT NULL and UNIQUE.
# Failure codes: NULL_PK, DUPLICATE_PK

passenger_flagged = passenger_typed.withColumn(
    "is_null_pk",
    when(col("pa_passengerid").isNull(), lit(True)).otherwise(lit(False)),
)

# Window-based duplicate detection
pk_window = Window.partitionBy("pa_passengerid")
passenger_flagged = passenger_flagged.withColumn(
    "_pk_count", _count("*").over(pk_window)
).withColumn(
    "is_duplicate_pk",
    when((col("pa_passengerid").isNotNull()) & (col("_pk_count") > 1), lit(True)).otherwise(lit(False)),
)

null_pk_count = passenger_flagged.filter(col("is_null_pk")).count()
duplicate_pk_count = passenger_flagged.filter(col("is_duplicate_pk")).count()

print(f"NULL_PK failures:      {null_pk_count}")
print(f"DUPLICATE_PK failures: {duplicate_pk_count}")

In [0]:
# ── Step 3: Referential Integrity Validation (§3.4) ──
# Rule: pa_nationkey must resolve to a valid n_nationkey in silver.nation.
# Failure code: INVALID_FK_NATION
#
# Parent-before-child order: nation is the parent of passenger.
# Uses silver.nation (already validated) as the reference.

valid_nation_keys = (
    spark.table("skyledger_airlines.silver.nation")
    .select(col("n_nationkey").cast("int").alias("n_nationkey"))
    .filter(col("n_nationkey").isNotNull())
    .distinct()
)

# Left join: rows with no match (or null pa_nationkey) get n_nationkey = null → invalid FK
passenger_flagged = passenger_flagged.join(
    valid_nation_keys,
    passenger_flagged["pa_nationkey"] == valid_nation_keys["n_nationkey"],
    "left",
).withColumn(
    "is_invalid_fk_nation",
    when(col("n_nationkey").isNull(), lit(True)).otherwise(lit(False)),
)

invalid_fk_count = passenger_flagged.filter(col("is_invalid_fk_nation")).count()
print(f"INVALID_FK_NATION failures: {invalid_fk_count}")

In [0]:
# ── Step 4: Build Rejection Reasons, Split Clean vs Quarantine, Write Tables ──

# Build comma-separated rejection_reason from all flag columns
passenger_flagged = passenger_flagged.withColumn(
    "rejection_reason",
    concat_ws(
        ", ",
        when(col("is_null_pk"), lit("NULL_PK")),
        when(col("is_duplicate_pk"), lit("DUPLICATE_PK")),
        when(col("is_invalid_fk_nation"), lit("INVALID_FK_NATION")),
    ),
)

# Clean records: passed every check (rejection_reason is empty string)
silver_passenger = passenger_flagged.filter(col("rejection_reason") == "").select(
    "pa_passengerid", "pa_name", "pa_address", "pa_nationkey",
    "pa_phone", "pa_loyaltypoints", "pa_travelpurpose", "pa_comment",
)

# Quarantined records: failed one or more checks
quarantine_passenger = passenger_flagged.filter(col("rejection_reason") != "").select(
    "pa_passengerid", "pa_name", "pa_address", "pa_nationkey",
    "pa_phone", "pa_loyaltypoints", "pa_travelpurpose", "pa_comment",
    "rejection_reason",
)

silver_count = silver_passenger.count()
quarantine_count = quarantine_passenger.count()

print(f"Silver rows:     {silver_count}")
print(f"Quarantine rows: {quarantine_count}")

# Write Silver table (overwrite for idempotency — same result on every run)
(silver_passenger.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("skyledger_airlines.silver.passenger"))

# Write Quarantine table (overwrite for idempotency)
(quarantine_passenger.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("skyledger_airlines.silver.passenger_quarantine"))

print("✅ Tables written: silver.passenger, silver.passenger_quarantine")

In [0]:
# ── Step 5: Bronze-Silver-Quarantine Reconciliation (§3.5) ──
# Bronze row count must equal Silver + Quarantine for the passenger table.

bronze_total = spark.table("skyledger_airlines.bronze.passenger").count()
silver_total = spark.table("skyledger_airlines.silver.passenger").count()
quarantine_total = spark.table("skyledger_airlines.silver.passenger_quarantine").count()

print(f"Bronze row count:      {bronze_total}")
print(f"Silver row count:      {silver_total}")
print(f"Quarantine row count:  {quarantine_total}")
print(f"Silver + Quarantine:   {silver_total + quarantine_total}")

assert bronze_total == silver_total + quarantine_total, (
    f"RECONCILIATION FAILED: Bronze ({bronze_total}) != "
    f"Silver ({silver_total}) + Quarantine ({quarantine_total})"
)

print("✅ Reconciliation PASSED: Bronze = Silver + Quarantine")

In [0]:
# ── Step 6: Data Quality Report ──
# Documents every check executed, even when zero rows fail.

dq_report = spark.createDataFrame(
    [
        ("passenger", "NULL_PK", "pa_passengerid must not be null", null_pk_count, "Quarantine"),
        ("passenger", "DUPLICATE_PK", "pa_passengerid must be unique across all rows", duplicate_pk_count, "Quarantine"),
        ("passenger", "INVALID_FK_NATION", "pa_nationkey must reference a valid n_nationkey in silver.nation", invalid_fk_count, "Quarantine"),
    ],
    ["table", "rule_code", "check_description", "failing_rows", "action_taken"],
)

display(dq_report)

# Show quarantined records if any exist
if quarantine_count > 0:
    print(f"\n── Quarantined Records ({quarantine_count} rows) ──")
    display(spark.table("skyledger_airlines.silver.passenger_quarantine"))
else:
    print("\nNo quarantined records — all passenger rows passed every check.")

# Show final Silver table
print(f"\n── Silver Passenger Table ({silver_count} rows) ──")
display(spark.table("skyledger_airlines.silver.passenger"))